In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
year = "2022"
month_1 = "08"
month_2 = "09"
day = "07"
ticker = "FBTP"

base_data_dir = Path("..") / "data" / "raw" / ticker
fbtp_fut_path_1 = base_data_dir / "futures" / year / month_1
fbtp_ctd_path_1 = base_data_dir / "ctd" / year / month_1
fbtp_fut_path_2 = base_data_dir / "futures" / year / month_2
fbtp_ctd_path_2 = base_data_dir / "ctd" / year / month_2

date_1 = f"{year}_{month_1}_08"
date_2 = f"{year}_{month_2}_{day}"
ctd_filename_1 = f"ctd_lob_freq_1s_{date_1}.parquet"
fut_filename_1 = f"futures_lob_freq_1s_{date_1}.parquet"
ctd_filename_2 = f"ctd_lob_freq_1s_{date_2}.parquet"
fut_filename_2 = f"futures_lob_freq_1s_{date_2}.parquet"

fut_df_1 = pd.read_parquet(fbtp_fut_path_1 / fut_filename_1, engine="fastparquet")
ctd_df_1 = pd.read_parquet(fbtp_ctd_path_1 / ctd_filename_1, engine="fastparquet")
fut_df_2 = pd.read_parquet(fbtp_fut_path_2 / fut_filename_2, engine="fastparquet")
ctd_df_2 = pd.read_parquet(fbtp_ctd_path_2 / ctd_filename_2, engine="fastparquet")

cf_df = pd.read_csv("../data/HCDT_IK1.csv", delimiter=";")
cf_df["Date"] = pd.to_datetime(cf_df["Date"], format="%d/%m/%Y").dt.date
cf_df["CF"] = cf_df["CF"].str.replace(",", ".").astype(float)

fut_df_1["MidPrice"] = (
    fut_df_1["MidPrice"] * cf_df.loc[cf_df["Date"] == fut_df_1.index[0].date(), "CF"].values[0]
)
fut_df_2["MidPrice"] = (
    fut_df_2["MidPrice"] * cf_df.loc[cf_df["Date"] == fut_df_2.index[0].date(), "CF"].values[0]
)

In [ ]:
rollover_dates = [
    "2010-03-08",
    "2010-06-08",
    "2010-09-08",
    "2010-12-08",
    "2011-03-08",
    "2011-06-08",
    "2011-09-08",
    "2011-12-08",
    "2012-03-08",
    "2012-06-07",
    "2012-09-06",
    "2012-12-06",
    "2013-03-07",
    "2013-06-06",
    "2013-09-06",
    "2013-12-06",
    "2014-03-06",
    "2014-06-06",
    "2014-09-08",
    "2014-12-08",
    "2015-03-06",
    "2015-06-08",
    "2015-09-08",
    "2015-12-08",
    "2016-03-08",
    "2016-06-08",
    "2016-09-08",
    "2016-12-08",
    "2017-03-08",
    "2017-06-08",
    "2017-09-07",
    "2017-12-07",
    "2018-03-08",
    "2018-06-07",
    "2018-09-06",
    "2018-12-06",
    "2019-03-07",
    "2019-06-06",
    "2019-09-06",
    "2019-12-06",
    "2020-03-06",
    "2020-06-08",
    "2020-09-08",
    "2020-12-08",
    "2021-03-08",
    "2021-06-08",
    "2021-09-08",
    "2021-12-08",
    "2022-03-08",
    "2022-06-08",
    "2022-09-08",
    "2022-12-08",
    "2023-03-08",
    "2023-06-08",
    "2023-09-07",
    "2023-12-07",
]

# Market opening and closing times

In [ ]:
path = "../data/raw/FBTP/futures/2022/08/"
files = os.listdir(path)
lob = pd.read_parquet(path + files[0], engine="fastparquet")
lob[lob.index == lob["MidPrice"].last_valid_index()]
lob.index[0]
lob.index[0].time()

In [ ]:
import os

ticker = "FBTP"
data_raw_path = Path("..") / "data" / "raw" / ticker
fut_path = data_raw_path / "futures"
ctd_path = data_raw_path / "ctd"

open_times = []
close_times = []
last_non_nan_times = []

for year in os.listdir(fut_path):
    if year != "2024":
        # print(f"Year: {year}")
        for month in os.listdir(fut_path / year):
            # print(f"    Month: {month}")
            fut_filenames = os.listdir(fut_path / year / month)
            for fut_filename in fut_filenames:
                # print(f"        {fut_filename}")
                fut_lob_df = pd.read_parquet(
                    fut_path / year / month / fut_filename, engine="fastparquet"
                )
                open_times.append(fut_lob_df.index[0].time())
                close_times.append(fut_lob_df.index[-1].time())
                fut_lob_df = fut_lob_df.drop(columns=["#RIC", "Domain", "GMT Offset", "Type"])
                mask = fut_lob_df.notna().any(axis=1)
                last_non_nan_times.append(fut_lob_df[mask].index[-1].time())

In [ ]:
# times_df = pd.DataFrame({"Open Time": open_times,
# "Close Time": close_times,
# "Last non NaN Time": last_non_nan_times})

open_time_s = pd.Series(open_times)
freq = open_time_s.value_counts().sort_index()
print(f"Unique open times: {len(open_time_s.unique())}")
plt.figure(figsize=(12, 6))
freq.plot(kind="bar")

close_time_s = pd.Series(close_times)
freq = close_time_s.value_counts().sort_index()
print(f"Unique close times: {len(close_time_s.unique())}")
plt.figure(figsize=(12, 6))
freq.plot(kind="bar")

last_non_nan_time_s = pd.Series(last_non_nan_times)
freq = last_non_nan_time_s.value_counts().sort_index()
print(f"Unique last non NaN times: {len(last_non_nan_time_s.unique())}")
plt.figure(figsize=(12, 6))
freq.plot(kind="bar")

# FBTP/FBTS vuoti

In [ ]:
ticker = "FBTS"
data_raw_path = Path("..") / "data" / "raw" / ticker
fut_path = data_raw_path / "futures"
ctd_path = data_raw_path / "ctd"

for year in os.listdir(fut_path):
    for month in os.listdir(fut_path / year):
        for fut_filename in os.listdir(fut_path / year / month):
            df = pd.read_parquet(fut_path / year / month / fut_filename, engine="fastparquet")
            # print(f"Number of rows in {fut_filename}: {len(df)}")